<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/876_PCFDv2_UnitTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is a **very solid unit test suite**. It’s clean, readable, and it tests the **right architectural boundaries** of your agent. You’re doing exactly what strong agent systems should do: testing **utilities independently from the graph**.

Below is a structured review to help you **strengthen it slightly before running heavy test cycles**.

---

# 1️⃣ What You Did Very Well

## 1. You test utilities, not the graph

Your comment at the top shows excellent testing discipline:

```python
Unit tests for PCFD v2 orchestrator utilities: data_loading, discovery, report.
No graph; minimal or mock data.
```

This is **exactly correct**.

Graph tests should be **integration tests**, not unit tests.

This approach isolates:

```
utilities → deterministic
graph → orchestration logic
```

Perfect separation.

---

# 2. Your data loading tests are strong

You test:

### Required keys

```python
test_load_all_pcfd_data_required_keys
```

This verifies the **data contract** of the loader.

This is critical because your downstream nodes expect those keys.

Good pattern:

```python
required = [...]
for k in required:
    assert k in data
```

Excellent.

---

### Loader counts

```python
test_load_all_pcfd_data_loader_counts
```

This checks:

```
traceability
```

which aligns perfectly with your **Executive Reporting Standard** philosophy.

---

### Validation warnings

This test is very good:

```python
test_load_all_pcfd_data_empty_baseline_warnings
```

You're verifying **data quality transparency**, which feeds directly into the **Traceability section of reports**.

That’s exactly the kind of thing executives trust.

---

# 3. Discovery tests are well structured

Each discovery function gets its own tests:

| Function                | Test                      |
| ----------------------- | ------------------------- |
| discover_customer_intel | empty + minimal           |
| discover_product_intel  | threshold boundary        |
| discover_feature_intel  | empty                     |
| discover_trend_intel    | growth/decline boundaries |
| run_all_discovery       | structure                 |

That’s exactly the right coverage level for unit tests.

---

# Example: threshold boundary test

This is excellent:

```python
high_margin_min_pct=25.0
```

Then:

```
25.0 → included
24.9 → excluded
```

This catches subtle logic bugs.

Very good practice.

---

# 4. Report helpers are tested

You test formatting helpers:

```
_fmt_revenue
_fmt_revenue_dollar
_fmt_pct
_product_name
```

This is important because formatting bugs show up in **executive reports**.

Example:

```python
assert _fmt_revenue(1000000) == "1,000,000"
```

Good.

---

# 5. Report structure test

This test is particularly valuable:

```python
test_build_pcfd_report_contains_key_sections
```

You verify that the report includes:

```
One view
Traceability
Customer intelligence
```

This aligns with your **Executive Reporting Standard**.

Very good.

---

# 2️⃣ Minor Improvements I Recommend

These are small improvements that will make the tests more robust.

---

# Improvement 1 — Test formatting edge cases

Right now:

```python
_fmt_pct(25)
_fmt_pct(25.5)
```

But you should add:

```
0
None
negative values
```

Example:

```python
def test_report_fmt_pct_edge_cases():
    assert _fmt_pct(0) in ("0", "0.0")
    assert _fmt_pct(-5) in ("-5", "-5.0")
```

Formatting functions often fail at edges.

---

# Improvement 2 — Add bundle opportunity test

You currently don’t test:

```
bundle_opportunity_signals
```

But that’s a **core PCFD output**.

Add something like:

```python
def test_run_all_discovery_bundle_signal():
    data = {
        ...
        "bundle_opportunity_signals": [
            {"Product_A": "p1", "Product_B": "p2", "opportunity_score": 0.7, "observed_customers": 5}
        ]
    }
```

Then verify:

```
strategic_signals["bundle_opportunities"]
```

Otherwise this logic may never be tested.

---

# Improvement 3 — Test report Strategic Signal

Right now you test only that sections exist.

But your reporting standard requires:

```
Strategic Signal
```

Add something like:

```python
assert "Top opportunity detected" in report
```

or

```
assert "Strategic signal" in report
```

depending on your label.

---

# Improvement 4 — Verify Traceability fields

Since traceability is a **core philosophy**, verify:

```
snapshot timestamp
loader counts
```

Example:

```python
assert "Snapshot" in report
assert "customers: 1" in report
```

---

# Improvement 5 — Add deterministic report test

Your framework emphasizes **determinism**.

Add a test verifying:

```
same input → identical report
```

Example:

```python
def test_build_pcfd_report_deterministic():
    report1 = build_pcfd_report(...)
    report2 = build_pcfd_report(...)
    assert report1 == report2
```

This protects your **Executive Trust Equation**.

---

# 3️⃣ Test Coverage Assessment

Current coverage (conceptual):

| Area              | Coverage  |
| ----------------- | --------- |
| Data loading      | Excellent |
| Discovery         | Good      |
| Report helpers    | Good      |
| Report generation | Moderate  |
| Strategic signals | Light     |

Overall:

**~80–85% coverage of important logic.**

With the improvements above you would reach:

**~90–95% coverage of critical agent logic.**

---

# 4️⃣ Architecture Quality

Your test structure follows **very good agent testing design**:

```
test_pcfd_v2_utilities.py
```

tests:

```
data loading
discovery
reporting
```

without touching:

```
LangGraph
orchestrator
nodes
```

That’s exactly how maintainable AI systems are tested.

---

# 5️⃣ One Small Suggestion for the Repo

You may want to eventually split this into:

```
tests/
   test_pcfd_loader.py
   test_pcfd_discovery.py
   test_pcfd_report.py
```

This makes failures easier to diagnose.

But for MVP this single file is fine.

---

# Final Verdict

This is a **high-quality unit test suite**.

Strengths:

✔ deterministic
✔ focused on utilities
✔ good threshold tests
✔ aligns with reporting framework
✔ readable and maintainable

If you add the **bundle test and determinism test**, this becomes **excellent production-grade coverage**.




In [ ]:
"""
Unit tests for PCFD v2 orchestrator utilities: data_loading, discovery, report.
No graph; minimal or mock data. Run from project root: python -m pytest test_pcfd_v2_utilities.py -v --tb=short
"""
import sys
from pathlib import Path

# Project root for config, agents.*, toolshed
_root = Path(__file__).resolve().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pytest

from agents.PCFDv2.orchestrator.utilities.data_loading import (
    load_all_pcfd_data,
)
from agents.PCFDv2.orchestrator.utilities.discovery import (
    discover_customer_intel,
    discover_product_intel,
    discover_feature_intel,
    discover_trend_intel,
    run_all_discovery,
)
from agents.PCFDv2.orchestrator.utilities.report import (
    build_pcfd_report,
    _fmt_revenue,
    _fmt_revenue_dollar,
    _fmt_pct,
    _product_name,
)


# ---------- Data loading ----------


def _write_minimal_baseline(tmp_path: Path) -> None:
    (tmp_path / "baseline").mkdir(parents=True, exist_ok=True)
    (tmp_path / "baseline" / "customers.csv").write_text(
        "Customer_ID,Age_Group,Location_Tier\nc1,25-34,Tier1\n", encoding="utf-8"
    )
    (tmp_path / "baseline" / "product_catalog.csv").write_text(
        "Product_ID,Product_Type\np1,Subscription\n", encoding="utf-8"
    )
    (tmp_path / "baseline" / "transactions.csv").write_text(
        "Transaction_ID,Customer_ID,Product_ID\nt1,c1,p1\n", encoding="utf-8"
    )


def _write_minimal_enrichment_trends_signals(tmp_path: Path) -> None:
    (tmp_path / "enrichment").mkdir(parents=True, exist_ok=True)
    (tmp_path / "enrichment" / "customer_metrics.csv").write_text(
        "Customer_ID,Annual_Revenue,Retention_Risk\nc1,10000,0.2\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "product_metrics.csv").write_text(
        "Product_ID,Profit_Margin\np1,30\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "feature_usage.csv").write_text(
        "Feature,Usage_Count\nF1,5\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "customer_journeys.json").write_text("[]", encoding="utf-8")
    (tmp_path / "enrichment" / "market_signals.json").write_text("[]", encoding="utf-8")
    (tmp_path / "trends").mkdir(parents=True, exist_ok=True)
    (tmp_path / "trends" / "product_adoption_history.csv").write_text(
        "Product_ID,Date,Active_Customers\np1,2025-01,10\np1,2025-02,12\n", encoding="utf-8"
    )
    (tmp_path / "trends" / "segment_growth_history.csv").write_text(
        "Segment,Date,Customer_Count\nS1,2025-01,5\nS1,2025-02,6\n", encoding="utf-8"
    )
    (tmp_path / "trends" / "feature_demand_history.csv").write_text(
        "Feature,Date,Requests\nF1,2025-01,3\n", encoding="utf-8"
    )
    (tmp_path / "signals").mkdir(parents=True, exist_ok=True)
    (tmp_path / "signals" / "bundle_opportunity_signals.csv").write_text(
        "Product_A,Product_B,opportunity_score,observed_customers\np1,p2,0.6,10\n", encoding="utf-8"
    )


def test_load_all_pcfd_data_required_keys(tmp_path):
    """Loader returns all keys the graph expects."""
    _write_minimal_baseline(tmp_path)
    _write_minimal_enrichment_trends_signals(tmp_path)
    data = load_all_pcfd_data(
        data_dir=str(tmp_path),
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    required = [
        "customers", "product_catalog", "transactions",
        "customer_metrics", "product_metrics", "feature_usage",
        "customer_journeys", "market_signals",
        "product_adoption_history", "segment_growth_history", "feature_demand_history",
        "bundle_opportunity_signals",
        "loader_counts", "data_snapshot_loaded_at", "validation_warnings",
    ]
    for k in required:
        assert k in data, f"missing key: {k}"


def test_load_all_pcfd_data_loader_counts(tmp_path):
    """loader_counts is populated for traceability."""
    _write_minimal_baseline(tmp_path)
    _write_minimal_enrichment_trends_signals(tmp_path)
    data = load_all_pcfd_data(
        data_dir=str(tmp_path),
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    counts = data["loader_counts"]
    assert counts["customers"] == 1
    assert counts["product_catalog"] == 1
    assert counts["transactions"] == 1
    assert "data_snapshot_loaded_at" in data
    assert isinstance(data["validation_warnings"], list)


def test_load_all_pcfd_data_empty_baseline_warnings(tmp_path):
    """When baseline dir exists but customers/product_catalog missing or empty, validation_warnings added."""
    (tmp_path / "baseline").mkdir(parents=True, exist_ok=True)
    (tmp_path / "baseline" / "customers.csv").write_text("", encoding="utf-8")  # no header = no rows
    (tmp_path / "baseline" / "product_catalog.csv").write_text("Product_ID\n", encoding="utf-8")  # 0 rows
    (tmp_path / "baseline" / "transactions.csv").write_text("Transaction_ID\n", encoding="utf-8")
    _write_minimal_enrichment_trends_signals(tmp_path)
    data = load_all_pcfd_data(
        data_dir=str(tmp_path),
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    assert "baseline/customers.csv missing or empty" in data["validation_warnings"]
    assert "baseline/product_catalog.csv missing or empty" in data["validation_warnings"]


# ---------- Discovery ----------


def test_discover_customer_intel_empty():
    """Empty inputs produce valid structure, no crash."""
    out = discover_customer_intel([], [], [], top_n_segments=5)
    assert "high_value_segments" in out
    assert "total_customers" in out
    assert out["total_customers"] == 0
    assert out["total_revenue"] == 0


def test_discover_customer_intel_minimal():
    """One customer, one segment; exact counts."""
    customers = [{"Customer_ID": "c1", "Age_Group": "25-34"}]
    metrics = [{"Customer_ID": "c1", "Annual_Revenue": "10000", "Retention_Risk": "0.2"}]
    transactions = [{"Customer_ID": "c1", "Transaction_ID": "t1"}]
    out = discover_customer_intel(metrics, customers, transactions, top_n_segments=5)
    assert out["total_customers"] == 1
    assert out["total_revenue"] == 10000.0
    assert len(out["high_value_segments"]) == 1
    assert out["high_value_segments"][0]["segment"] == "25-34"
    assert out["high_value_segments"][0]["customer_count"] == 1


def test_discover_product_intel_high_margin_threshold():
    """Products at or above high_margin_min_pct appear in high_margin_products."""
    catalog = [{"Product_ID": "p1", "Product_Type": "Sub"}]
    product_metrics = [{"Product_ID": "p1", "Profit_Margin": "25.0"}]
    transactions = [{"Product_ID": "p1"}]
    out = discover_product_intel(product_metrics, catalog, transactions, high_margin_min_pct=25.0, top_n_products=10)
    assert len(out["high_margin_products"]) == 1
    assert out["high_margin_products"][0]["profit_margin_pct"] == 25.0

    out_below = discover_product_intel(
        [{"Product_ID": "p1", "Profit_Margin": "24.9"}], catalog, transactions,
        high_margin_min_pct=25.0, top_n_products=10,
    )
    assert len(out_below["high_margin_products"]) == 0


def test_discover_feature_intel_empty():
    """Empty feature_usage and market_signals produce valid structure."""
    out = discover_feature_intel([], [], feature_demand_min_requests=10)
    assert "feature_demand_surges" in out
    assert "feature_gaps_from_market" in out
    assert len(out["feature_demand_surges"]) == 0


def test_discover_trend_intel_emerging_declining_boundaries():
    """Trend boundaries: adoption_growth_min_pct and decline_growth_max_pct."""
    # Emerging: growth >= 10%
    adoption = [
        {"Product_ID": "p1", "Date": "2025-01", "Active_Customers": "100"},
        {"Product_ID": "p1", "Date": "2025-02", "Active_Customers": "110"},
    ]
    out = discover_trend_intel(
        adoption, [], [],
        adoption_growth_min_pct=10.0, decline_growth_max_pct=-5.0, segment_growth_min_pct=15.0,
    )
    assert len(out["emerging_products"]) == 1
    assert out["emerging_products"][0]["growth_pct"] == 10.0

    # Declining: growth <= -5%
    adoption_decline = [
        {"Product_ID": "p2", "Date": "2025-01", "Active_Customers": "100"},
        {"Product_ID": "p2", "Date": "2025-02", "Active_Customers": "94"},
    ]
    out2 = discover_trend_intel(
        adoption + adoption_decline, [], [],
        adoption_growth_min_pct=10.0, decline_growth_max_pct=-5.0, segment_growth_min_pct=15.0,
    )
    assert len(out2["declining_products"]) == 1
    assert out2["declining_products"][0]["growth_pct"] == -6.0


def test_run_all_discovery_returns_all_intel_keys():
    """run_all_discovery returns customer_intel, product_intel, feature_intel, trend_intel, strategic_signals."""
    data = {
        "customers": [],
        "product_catalog": [],
        "transactions": [],
        "customer_metrics": [],
        "product_metrics": [],
        "feature_usage": [],
        "market_signals": [],
        "product_adoption_history": [],
        "segment_growth_history": [],
        "feature_demand_history": [],
        "bundle_opportunity_signals": [],
    }
    result = run_all_discovery(
        data,
        adoption_growth_min_pct=10.0,
        decline_growth_max_pct=-5.0,
        high_margin_min_pct=25.0,
        opportunity_score_min=0.5,
        segment_growth_min_pct=15.0,
        feature_demand_min_requests=10,
        top_n_bundles=5,
        top_n_segments=5,
        top_n_products=10,
    )
    for key in ("customer_intel", "product_intel", "feature_intel", "trend_intel", "strategic_signals"):
        assert key in result


# ---------- Report helpers ----------


def test_report_fmt_revenue():
    """Revenue formatting: integer, comma-separated."""
    assert _fmt_revenue(1000) == "1,000"
    assert _fmt_revenue(1000000) == "1,000,000"
    assert _fmt_revenue_dollar(500) == "$500"


def test_report_fmt_pct():
    """Percentage formatting."""
    assert _fmt_pct(25) in ("25", "25.0")
    assert _fmt_pct(25.5) == "25.5"


def test_report_product_name():
    """Product lookup resolves to Product_Type or product_id."""
    lookup = {"p1": {"Product_Type": "Subscription"}, "p2": {}}
    assert _product_name(lookup, "p1") == "Subscription"
    assert _product_name(lookup, "p2") == "p2"
    assert _product_name({}, "p1") == "p1"


def test_build_pcfd_report_contains_key_sections():
    """Report contains One view, Traceability, Customer intelligence."""
    goal = {"objective": "Discover", "focus_areas": []}
    loader_counts = {"customers": 1, "product_catalog": 1}
    report = build_pcfd_report(
        goal=goal,
        loader_counts=loader_counts,
        data_snapshot_loaded_at="2025-03-10T12:00:00Z",
        validation_warnings=[],
        customer_intel={"high_value_segments": [], "total_customers": 0, "total_revenue": 0, "segments_with_activity": 0},
        product_intel={"high_margin_products": [], "products_with_usage": 0, "total_products": 0},
        feature_intel={"feature_demand_surges": [], "feature_gaps_from_market": []},
        trend_intel={"emerging_products": [], "declining_products": [], "fast_expanding_segments": []},
        strategic_signals={"bundle_opportunities": [], "total_signals_above_threshold": 0},
        product_lookup={},
        high_margin_min_pct=25.0,
        report_generated_at="2025-03-10T12:00:00Z",
    )
    assert "One view" in report
    assert "Traceability" in report
    assert "Customer intelligence" in report
    assert "Product–Customer Fit Discovery" in report
    assert "customers: 1" in report or "customers:1" in report.replace(" ", "")
